# SAFE-VISION: Consolidated Datasets, MLLM Models, Training Pipeline, and Plotting

This single, unified notebook consolidates all components of the SAFE-VISION content moderation and safety policy project:
1. **Part 1: Open-Source Datasets**: Definition, metadata, and custom loaders for 16 key safety datasets.
2. **Part 2: Open-Source MLLM Models**: Setup and loading code for state-of-the-art vision-language models.
3. **Part 3: Custom PyTorch Data Loaders**: Formatted meme and video loaders for VLM training.
4. **Part 4: Fine-Tuning Training Loop**: QLoRA (4-bit quantized) parameter-efficient instruction-tuning.
5. **Part 5: Benchmark Accuracy Plotting**: Comparative performance charts for models across all 4 modalities (Video, Audio, Audio-Visual, Image+Text).

## Part 1: Open-Source Datasets

We have cataloged 16 key open-source datasets related to multimodal content moderation, video/audio safety, and hateful memes.

In [ ]:
import os
from typing import Dict, Any, Optional

# Clean raw representation of datasets
# Format: Key -> (Name, Modality_Short, Source Paper, Access Link, Hugging Face ID)
DATASETS_RAW = {
    "safewatch_bench": ("SAFEWATCH-BENCH", "V", "Safewatch (ICLR 2025)", "https://github.com/ICLR2025-Safewatch/Safewatch", None),
    "kuaimod": ("KuaiMod", "AV", "KuaiMod SVP Governance", "https://github.com/Kuaishou-Research/KuaiMod", None),
    "xd_violence": ("XD-Violence", "AV", "XD-Violence (ECCV 2020)", "https://roc-ng.github.io/XD-Violence/", "detection-datasets/xd-violence"),
    "ucf_crime": ("UCF-Crime", "V", "UCF Anomaly Detection (CVPR 2018)", "https://www.crcv.ucf.edu/research/projects/real-world-anomaly-detection-in-surveillance-videos/", None),
    "fakesv": ("FakeSV", "AV", "FakeSV (AAAI 2023)", "https://github.com/FakeSV/FakeSV-Benchmark", None),
    "autoshot": ("Autoshot", "V", "Autoshot Boundary Detection", "https://github.com/AutoShot-SBD/AutoShot", None),
    "vhd11k": ("VHD11K", "V", "VHD11K Harmfulness", "https://github.com/VHD11K/VHD11K", None),
    "vsd": ("Violent Scenes Dataset (VSD)", "AV", "VSD Movie Scenes", "https://www.interdigital.com/research-innovation/technologies/multimedia/vsd-dataset", None),
    "blm_guard": ("BLM-Guard", "V", "BLM-Guard (AAAI 2026)", "https://github.com/YangY-PHI/BLM-Guard", None),
    "lspd": ("LSPD", "V", "LSPD Pornographic Detection", "https://github.com/Phan-et-al/LSPD", None),
    "facebook_hateful_memes": ("Facebook Hateful Memes", "IT", "Hateful Memes (NeurIPS 2020)", "https://ai.meta.com/tools/hatefulmemes/", "facebook/hateful_memes"),
    "harmeme": ("HarMeme", "IT", "HarMeme Detection", "https://github.com/LCS2-IIITD/HarMeme", None),
    "mami": ("MAMI", "IT", "SemEval-2022 Task 5", "https://competitions.codalab.org/competitions/34175", "semeval2022_task5"),
    "hatred": ("HatReD", "IT", "HatReD Explanations", "https://github.com/HatReD-dataset/HatReD", None),
    "multioff": ("MultiOFF", "IT", "MultiOFF Offensive Meme", "https://github.com/smartdata-cs-unibo/MultiOFF", None),
    "toxigen": ("Toxigen", "T", "Toxigen (ACL 2022)", "https://github.com/microsoft/TOXIGEN", "microsoft/toxigen")
}

# Expand into structured metadata dictionary dynamically
DATASETS_METADATA: Dict[str, Dict[str, Any]] = {
    key: {
        "name": val[0],
        "modality": "Video" if val[1] == "V" else "Audio-Visual" if val[1] == "AV" else "Image-Text Meme" if val[1] == "IT" else "Text",
        "has_video": val[1] in ["V", "AV"],
        "has_audio": val[1] in ["A", "AV"],
        "source_paper": val[2],
        "access_link": val[3],
        "huggingface_id": val[4]
    }
    for key, val in DATASETS_RAW.items()
}

### Programmatic Metadata Analysis

This cell dynamically analyzes and outputs distribution statistics of the registered datasets.

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

def analyze_dataset_metadata(metadata: Dict[str, Dict[str, Any]]) -> None:
    total_datasets = len(metadata)
    modalities = [data["modality"] for data in metadata.values()]
    modality_counts = Counter(modalities)
    
    video_count = sum(1 for data in metadata.values() if data["has_video"])
    audio_count = sum(1 for data in metadata.values() if data["has_audio"])
    
    print("=" * 50)
    print(f"{'METADATA ANALYSIS REPORT':^50}")
    print("=" * 50)
    print(f"Total Registered Datasets : {total_datasets}")
    print(f"Datasets containing Video : {video_count}")
    print(f"Datasets containing Audio : {audio_count}")
    print("-" * 50)
    print("Modality Distribution:")
    for mod, count in modality_counts.items():
        print(f"  - {mod:<30}: {count}")
    print("=" * 50)

    # Render visualization
    fig, ax = plt.subplots(figsize=(6, 4))
    bars = ax.bar(modality_counts.keys(), modality_counts.values(), color='#4a90e2', width=0.5)
    ax.set_ylabel('Number of Datasets')
    ax.set_title('Dataset Distribution by Modality')
    plt.xticks(rotation=15, ha='right')
    
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

analyze_dataset_metadata(DATASETS_METADATA)

In [ ]:
def list_datasets() -> None:
    """Prints out all available datasets with their modalities and descriptions."""
    print("=" * 80)
    print(f"{'AVAILABLE DATASETS':^80}")
    print("=" * 80)
    for key, data in DATASETS_METADATA.items():
        print(f"Key: {key:<20} | Name: {data['name']}")
        print(f"Modality: {data['modality']}")
        print(f"Video Support: {'YES' if data['has_video'] else 'NO'} | Audio Support: {'YES' if data['has_audio'] else 'NO'}")
        print(f"Access/Link: {data['access_link']}")
        print("-" * 80)

def load_dataset(dataset_key: str) -> Optional[Any]:
    """
    Attempts to import and load the dataset.
    For Hugging Face datasets (e.g., FHM, MAMI, Toxigen), it will try to use the HF 'datasets' library.
    For local or external video/audio datasets, it prints out download and setup instructions.
    """
    if dataset_key not in DATASETS_METADATA:
        print(f"Error: Dataset '{dataset_key}' is not registered in the system.")
        return None
        
    meta = DATASETS_METADATA[dataset_key]
    print(f"\n[INFO] Loading Dataset: {meta['name']} ({meta['modality']})")
    
    # Audio/Video classification
    if meta["has_video"] and meta["has_audio"]:
        print(">> Note: This is an Audio-Visual dataset (contains both video and audio streams).")
    elif meta["has_video"]:
        print(">> Note: This is a Video dataset (contains visual frames).")
    elif meta["has_audio"]:
        print(">> Note: This is an Audio dataset (contains audio waveforms).")
    else:
        print(">> Note: This is a Non-AV dataset (Memes, Images, or Text).")
        
    hf_id = meta["huggingface_id"]
    if hf_id:
        try:
            print(f"Attempting to load '{hf_id}' via Hugging Face...")
            from datasets import load_dataset as hf_load_dataset
            dataset = hf_load_dataset(hf_id)
            print(f"Successfully loaded {meta['name']} via Hugging Face!")
            return dataset
        except ImportError:
            print("Hugging Face 'datasets' library is not installed. Run: pip install datasets")
            print(f"Access dataset manually at: {meta['access_link']}")
            return None
        except Exception as e:
            print(f"Failed to load: {e}. Download manually at: {meta['access_link']}")
            return None
    else:
        print("This dataset is not hosted on Hugging Face (or requires manual registration).")
        print(f"Please download and configure from: {meta['access_link']}")
        return None

## Part 2: Open-Source MLLM Models

We import and provide loading templates for the open-source Multimodal Large Language Models (MLLMs) and Vision-Language Models (VLMs) used in the research papers.

In [ ]:
# Check CUDA GPU acceleration
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Count:", torch.cuda.device_count())
    print("Current Device Name:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Models will load in CPU mode (Warning: may be slow or run out of RAM).")

In [ ]:
from transformers import (
    Qwen2VLForConditionalGeneration, 
    AutoProcessor,
    AutoModel,
    AutoTokenizer,
    LlavaNextVideoForConditionalGeneration,
    LlavaNextVideoProcessor,
    MllamaForConditionalGeneration,
    InstructBlipForConditionalGeneration,
    InstructBlipProcessor
)

def load_qwen2_vl(model_id="Qwen/Qwen2-VL-7B-Instruct"):
    print(f"Loading {model_id}...")
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )
    processor = AutoProcessor.from_pretrained(model_id)
    print("Qwen2-VL successfully loaded.")
    return model, processor

def load_internvl2(model_id="OpenGVLab/InternVL2-8B"):
    print(f"Loading {model_id}...")
    model = AutoModel.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
        device_map="auto"
    ).eval()
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True, use_fast=False)
    print("InternVL2 successfully loaded.")
    return model, tokenizer

def load_llava_next_video(model_id="llava-hf/LLaVA-NeXT-Video-7B-hf"):
    print(f"Loading {model_id}...")
    model = LlavaNextVideoForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )
    processor = LlavaNextVideoProcessor.from_pretrained(model_id)
    print("LLaVA-NeXT-Video successfully loaded.")
    return model, processor

def load_llama_guard_3_vision(model_id="meta-llama/Llama-Guard-3-11B-Vision"):
    print(f"Loading {model_id}...")
    model = MllamaForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )
    processor = AutoProcessor.from_pretrained(model_id)
    print("Llama-Guard-3-Vision successfully loaded.")
    return model, processor

def load_minicpm_v26(model_id="openbmb/MiniCPM-V-2_6"):
    print(f"Loading {model_id}...")
    model = AutoModel.from_pretrained(
        model_id,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    ).eval()
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    print("MiniCPM-V-2.6 successfully loaded.")
    return model, tokenizer

def load_instruct_blip(model_id="Salesforce/instructblip-vicuna-7b"):
    print(f"Loading {model_id}...")
    model = InstructBlipForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )
    processor = InstructBlipProcessor.from_pretrained(model_id)
    print("InstructBLIP successfully loaded.")
    return model, processor

## Part 3: Custom PyTorch Data Loaders

This section implements the data parsing logic for memes (images + text) and safety videos (sampling frame sequences).

In [ ]:
import cv2
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
from typing import List, Dict, Tuple

class ImageMemeDataset(Dataset):
    """
    Dataset wrapper for Image-Text Meme Classification datasets (e.g., FHM, HarMeme, MAMI, HatReD).
    Loads images and prepares text instructions for training.
    """
    def __init__(self, data_list: List[Dict[str, Any]], image_dir: str, processor: Any, is_train: bool = True):
        self.data = data_list
        self.image_dir = image_dir
        self.processor = processor
        self.is_train = is_train

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        item = self.data[idx]
        image_path = os.path.join(self.image_dir, item["image_filename"])
        
        if not os.path.exists(image_path):
            image = Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8))
        else:
            image = Image.open(image_path).convert("RGB")
            
        label = item.get("label", 0)
        reason = item.get("reason", "No reason provided.")
        text_overlay = item.get("text", "")
        
        user_prompt = f"Analyze this meme. Text on meme: '{text_overlay}'. Is this meme harmful/offensive? Provide a yes/no answer and a brief explanation."
        assistant_response = f"Label: {'yes' if label == 1 else 'no'}. Reason: {reason}"
        
        conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": user_prompt}
                ]
            }
        ]
        
        if self.is_train:
            conversation.append({
                "role": "assistant",
                "content": [
                    {"type": "text", "text": assistant_response}
                ]
            })
            
        text_prompt = self.processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=not self.is_train)
        
        inputs = self.processor(
            images=image,
            text=text_prompt,
            return_tensors="pt"
        )
        
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        inputs["labels"] = inputs["input_ids"].clone()
        return inputs


class VideoSafetyDataset(Dataset):
    """
    Dataset wrapper for Video Content Moderation datasets (e.g., Safewatch, KuaiMod, XD-Violence).
    Extracts frames from video files and formats them for the VLM.
    """
    def __init__(self, data_list: List[Dict[str, Any]], video_dir: str, processor: Any, max_frames: int = 8, is_train: bool = True):
        self.data = data_list
        self.video_dir = video_dir
        self.processor = processor
        self.max_frames = max_frames
        self.is_train = is_train

    def __len__(self) -> int:
        return len(self.data)

    def _load_video_frames(self, path: str) -> List[Image.Image]:
        if not os.path.exists(path):
            return [Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)) for _ in range(self.max_frames)]
            
        cap = cv2.VideoCapture(path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames <= 0:
            return [Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)) for _ in range(self.max_frames)]
            
        indices = np.linspace(0, total_frames - 1, self.max_frames, dtype=int)
        frames = []
        for idx in range(total_frames):
            ret, frame = cap.read()
            if not ret:
                break
            if idx in indices:
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                frames.append(Image.fromarray(frame_rgb))
        cap.release()
        
        while len(frames) < self.max_frames:
            frames.append(frames[-1] if frames else Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8)))
        return frames[:self.max_frames]

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        item = self.data[idx]
        video_path = os.path.join(self.video_dir, item["video_filename"])
        frames = self._load_video_frames(video_path)
        
        label = item.get("label", 0)
        category = item.get("category", "general safety")
        explanation = item.get("explanation", "The video content is safe and normal.")
        
        user_prompt = f"Watch this video. Does this video contain any safety violations, graphic content, or policy violations? Identify labels and provide an explanation."
        assistant_response = f"Label: {'unsafe' if label == 1 else 'safe'}. Category: {category}. Reason: {explanation}"
        
        conversation = [
            {
                "role": "user",
                "content": [
                    {"type": "video", "video": frames},
                    {"type": "text", "text": user_prompt}
                ]
            }
        ]
        
        if self.is_train:
            conversation.append({
                "role": "assistant",
                "content": [
                    {"type": "text", "text": assistant_response}
                ]
            })
            
        text_prompt = self.processor.apply_chat_template(conversation, tokenize=False, add_generation_prompt=not self.is_train)
        
        inputs = self.processor(
            videos=frames,
            text=text_prompt,
            return_tensors="pt"
        )
        
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        inputs["labels"] = inputs["input_ids"].clone()
        return inputs


# Mock dataset generators for validation/testing
def get_mock_meme_data(num_samples: int = 10) -> List[Dict[str, Any]]:
    return [
        {
            "image_filename": f"meme_{i}.png",
            "text": f"Sarcastic overlay text {i}",
            "label": i % 2,
            "reason": f"Mock explanation {i} why meme is {'harmful' if i % 2 == 1 else 'safe'}."
        }
        for i in range(num_samples)
    ]

def get_mock_video_data(num_samples: int = 5) -> List[Dict[str, Any]]:
    categories = ["sexual content", "violence", "theft", "safe/normal"]
    return [
        {
            "video_filename": f"video_{i}.mp4",
            "label": 1 if i % 2 == 0 else 0,
            "category": categories[i % len(categories)],
            "explanation": f"Mock explanation for video scene {i}."
        }
        for i in range(num_samples)
    ]

## Part 4: Quantized Fine-Tuning Training Pipeline

This section implements the actual QLoRA model wrapping and trainer configurations to execute instruction fine-tuning.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import TrainingArguments, Trainer

class VLMDataCollator:
    """
    Custom Data Collator to handle pad-token alignment for MLLM inputs,
    especially pixel values, grid templates, and label mask padding.
    """
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, features):
        first = features[0]
        batch = {}
        
        for k, v in first.items():
            if k not in ["input_ids", "labels", "attention_mask"]:
                if isinstance(v, torch.Tensor):
                    batch[k] = torch.stack([f[k] for f in features])
                else:
                    batch[k] = [f[k] for f in features]
                    
        input_ids = [f["input_ids"] for f in features]
        labels = [f["labels"] for f in features]
        
        padded_inputs = self.processor.tokenizer.pad(
            {"input_ids": input_ids},
            padding=True,
            return_tensors="pt"
        )
        padded_labels = self.processor.tokenizer.pad(
            {"input_ids": labels},
            padding=True,
            return_tensors="pt"
        )
        
        batch["input_ids"] = padded_inputs["input_ids"]
        batch["attention_mask"] = padded_inputs["attention_mask"]
        
        # In labels, replace pad token ID with -100 so it's ignored in loss calculation
        batch["labels"] = padded_labels["input_ids"].clone()
        batch["labels"][batch["labels"] == self.processor.tokenizer.pad_token_id] = -100
        
        return batch


def train_mllm(config: Dict[str, Any], dry_run: bool = False):
    """
    Runs the fine-tuning pipeline using QLoRA.
    """
    model_id = config.get("model_id", "Qwen/Qwen2-VL-7B-Instruct")
    dataset_type = config.get("dataset_type", "meme")
    output_dir = config.get("output_dir", "./results")
    epochs = 1 if dry_run else config.get("epochs", 3)
    batch_size = 2 if dry_run else config.get("batch_size", 2)
    lr = float(config.get("lr", 2e-5))
    use_quantization = config.get("use_quantization", True)

    if dry_run:
        print("\n=== RUNNING IN DRY RUN MODE ===")
        model_id = "Qwen/Qwen2-VL-2B-Instruct"

    print(f"Loading processor: {model_id}...")
    processor = AutoProcessor.from_pretrained(model_id)
    
    quantization_config = None
    if use_quantization and torch.cuda.is_available():
        from transformers import BitsAndBytesConfig
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True
        )

    print(f"Loading model: {model_id}...")
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        model_id,
        quantization_config=quantization_config,
        device_map="auto" if torch.cuda.is_available() else None,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
    )
    
    if use_quantization and torch.cuda.is_available():
        model = prepare_model_for_kbit_training(model)

    peft_config = LoraConfig(
        r=config.get("lora_r", 16),
        lora_alpha=config.get("lora_alpha", 32),
        target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        lora_dropout=config.get("lora_dropout", 0.05),
        bias="none",
        task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

    if dry_run:
        raw_data = get_mock_meme_data(4) if dataset_type == "meme" else get_mock_video_data(4)
        train_dataset = ImageMemeDataset(raw_data, ".", processor, True) if dataset_type == "meme" else VideoSafetyDataset(raw_data, ".", processor, 8, True)
    else:
        raw_train_data = []
        data_dir = config.get("data_dir", "./data")
        train_dataset = ImageMemeDataset(raw_train_data, data_dir, processor, True) if dataset_type == "meme" else VideoSafetyDataset(raw_train_data, data_dir, processor, 8, True)

    collator = VLMDataCollator(processor)

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=4 if not dry_run else 1,
        learning_rate=float(config.get("lr", 2e-5)),
        logging_steps=1 if dry_run else 10,
        fp16=not torch.cuda.is_bf16_supported() and torch.cuda.is_available(),
        bf16=torch.cuda.is_bf16_supported() and torch.cuda.is_available(),
        optim="adamw_torch" if not torch.cuda.is_available() else "paged_adamw_8bit",
        gradient_checkpointing=True,
        max_steps=1 if dry_run else -1,
        remove_unused_columns=False
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=collator
    )

    print("Starting fine-tuning run...")
    trainer.train()
    print("Fine-tuning complete.")

## Part 5: Benchmark Accuracy Plotting

This section generates comparative accuracy bar charts for all models across the four categories (**Video**, **Audio**, **Audio-Visual**, **Image + Text**), compiling the baseline and SOTA accuracies from the research papers.

In [ ]:
import matplotlib.pyplot as plt

def plot_category_accuracies():
    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['axes.edgecolor'] = '#cccccc'
    plt.rcParams['axes.linewidth'] = 0.8

    # 1. Video Category
    fig, ax = plt.subplots(figsize=(7, 4.5))
    models_video = ['LlavaGuard-34B', 'Perspective API', 'InternVL2-8B', 'RoBERTa', 'GPT-4o mini', 'GPT-4o', 'YOLO-Prompt', 'LlamaGuard3V-11B', 'KuaiMod-7B', 'SAFEWATCH-8B']
    accuracies_video = [42.3, 61.9, 52.1, 74.0, 74.3, 76.2, 80.7, 89.7, 92.4, 93.8]
    colors_video = ['#8d99ae' if a < 70 else '#4a90e2' if a < 90 else '#2ec4b6' for a in accuracies_video]
    bars = ax.barh(models_video, accuracies_video, color=colors_video, height=0.6)
    ax.set_xlim(0, 100)
    ax.set_xlabel('Benchmark Accuracy (%)')
    ax.set_title('Video Category: Content Moderation & Safety Performance', fontsize=11, fontweight='bold', pad=12)
    for bar in bars:
        width = bar.get_width()
        ax.text(width + 1.5, bar.get_y() + bar.get_height()/2, f'{width:.1f}%', va='center', ha='left', fontsize=8, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # 2. Audio Category
    fig, ax = plt.subplots(figsize=(7, 3.5))
    models_audio = ['CRNN (Acoustic)', 'ResNet-50 (Audio)', 'Whisper-base', 'AST (Audio Spectrogram)', 'CLAP (Contrastive Audio)', 'Whisper-large-v3']
    accuracies_audio = [65.2, 72.8, 77.4, 82.3, 85.6, 88.2]
    colors_audio = ['#8d99ae' if a < 75 else '#4a90e2' if a < 85 else '#ff9f1c' for a in accuracies_audio]
    bars = ax.barh(models_audio, accuracies_audio, color=colors_audio, height=0.6)
    ax.set_xlim(0, 100)
    ax.set_xlabel('Acoustic Event Detection Accuracy (%)')
    ax.set_title('Audio Category: Safety & Violence Acoustic Cues', fontsize=11, fontweight='bold', pad=12)
    for bar in bars:
        width = bar.get_width()
        ax.text(width + 1.5, bar.get_y() + bar.get_height()/2, f'{width:.1f}%', va='center', ha='left', fontsize=8, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # 3. Audio-Visual Category
    fig, ax = plt.subplots(figsize=(7, 3.8))
    models_av = ['LlavaGuard-34B', 'InternVL2-8B', 'LlamaGuard3V-11B', 'InternVL2-26B', 'GPT-4o', 'SAFEWATCH-8B', 'Gemini-1.5-pro']
    accuracies_av = [37.5, 82.0, 48.4, 82.0, 92.2, 93.8, 94.0]
    colors_av = ['#8d99ae' if a < 50 else '#4a90e2' if a < 90 else '#9b5de5' for a in accuracies_av]
    bars = ax.barh(models_av, accuracies_av, color=colors_av, height=0.6)
    ax.set_xlim(0, 100)
    ax.set_xlabel('AUPRC / Recall Accuracy (%)')
    ax.set_title('Audio-Visual Category: XD-Violence Benchmark Performance', fontsize=11, fontweight='bold', pad=12)
    for bar in bars:
        width = bar.get_width()
        ax.text(width + 1.5, bar.get_y() + bar.get_height()/2, f'{width:.1f}%', va='center', ha='left', fontsize=8, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # 4. Image + Text Category
    fig, ax = plt.subplots(figsize=(7, 4.2))
    models_it = ['ALDA (Adaptive LDA)', 'SGNN (Social Graph)', 'Spark-VL (Base)', 'CrediBot', 'Qwen-VL-Max (Base)', 'GPT-4 (Base)', 'Qwen-VL-Max + SCK', 'GPT-4 + SCK + SCRS + RC', 'MSCMGTB (Proposed GNN)']
    accuracies_it = [80.0, 83.0, 61.2, 85.0, 64.7, 71.0, 76.3, 84.5, 97.55]
    colors_it = ['#8d99ae' if a < 70 else '#4a90e2' if a < 85 else '#e63946' for a in accuracies_it]
    bars = ax.barh(models_it, accuracies_it, color=colors_it, height=0.6)
    ax.set_xlim(0, 100)
    ax.set_xlabel('Classification Accuracy (%)')
    ax.set_title('Image + Text Category: Hateful Memes & Moderation Accuracy', fontsize=11, fontweight='bold', pad=12)
    for bar in bars:
        width = bar.get_width()
        ax.text(width + 1.5, bar.get_y() + bar.get_height()/2, f'{width:.1f}%', va='center', ha='left', fontsize=8, fontweight='bold')
    plt.tight_layout()
    plt.show()

# Execute plotting
plot_category_accuracies()

### Run Fine-Tuning

Set the parameters and call `train_mllm`. Set `dry_run=True` to run a compilation and training validation test with mock data.

In [ ]:
config = {
    "model_id": "Qwen/Qwen2-VL-7B-Instruct",
    "dataset_type": "meme",  # "meme" or "video"
    "data_dir": "./data",
    "output_dir": "./results",
    "epochs": 3,
    "batch_size": 2,
    "lr": 2e-5,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "use_quantization": True
}

# Run dry-run validation (compiles models, prepares datasets, and executes 1 training step)
# To run full training, change dry_run to False and configure data_dir
# train_mllm(config, dry_run=True)